In [1]:
# Importing Modules
import pandas as pd
from src.utils.smiles2morganfp import MorganFP
from src.model import ML_Models
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"])
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"])
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

######################## DATA-2 ##################################
# Loading FreeSolv data
freeSolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freeSolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

# Generate FreeSolv FP
freeSolv_train_fp = MorganFP(freeSolv_train_data["smiles"])
freeSolv_train_fp["smiles"] = freeSolv_train_fp.index
freeSolv_train_fp = freeSolv_train_fp.merge(freeSolv_train_data, on="smiles")
freeSolv_test_fp = MorganFP(freeSolv_test_data["smiles"])
freeSolv_test_fp["smiles"] = freeSolv_test_fp.index
freeSolv_test_fp = freeSolv_test_fp.merge(freeSolv_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"])
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"])
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

In [3]:
# Function to run ML training and testing
def RunML(model, train_data, test_data, dataName, modelName):
    
	# Training data
	X_train, y_train = train_data.drop(["smiles", "target"], axis=1), train_data["target"]
    
    # Test (hold-out) data
	X_test, y_test = test_data.drop(["smiles", "target"], axis=1), test_data["target"]

	# Model training
	model = model.fit(X_train.to_numpy(), y_train)

	# Model testing
	y_pred = model.predict(X_test.to_numpy())

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
	# Calculate pearson r
	r, p_val = pearsonr(y_test, y_pred)
    
	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)], 
			"r":[round(r, 2)], "p-val":[round(p_val, 3)]})

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_fp, "test":esol_test_fp},
         "FreeSolv":{"train":freeSolv_train_fp, "test":freeSolv_test_fp},
          "Lipophilicity":{"train":lipophilicity_train_fp, "test":lipophilicity_test_fp}}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in ML_Models.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        temp_out.append(RunML(model, data["train"], data["test"], dataName, modelName))

# Final output
ML_out = pd.concat(temp_out, ignore_index=True)
ML_out.to_csv("results/Output_ML_Analysis.csv")
ML_out

,Data,Model,MAE,RMSE,r,p-val
0,ESOL,LR,1.50,2.16,0.64,0.0
1,FreeSolv,LR,1.38,2.20,0.84,0.0
2,Lipophilicity,LR,0.92,1.28,0.58,0.0
3,ESOL,ENet,0.89,1.15,0.85,0.0
4,FreeSolv,ENet,1.28,2.07,0.87,0.0
5,Lipophilicity,ENet,0.69,0.87,0.71,0.0
6,ESOL,SVM,0.72,1.00,0.89,0.0
7,FreeSolv,SVM,1.23,2.01,0.88,0.0
8,Lipophilicity,SVM,0.52,0.71,0.82,0.0
9,ESOL,RF,0.90,1.17,0.85,0.0


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    sns.set_theme(style="ticks", context="paper")
    g = sns.catplot(data=data, kind="bar", x="Model", y=target, hue="Model",
                    col="Data", col_wrap=3, sharey=False, height=4, width=0.8,
                    aspect=0.8, dodge=False)
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/ML_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(ML_out, "RMSE")
# Plotting barplot for MAE
plot_bar(ML_out, "MAE")
# Plotting barplot for 
plot_bar(ML_out, "r")